# TP 2 Transformer -> Construire l'auto-attention pas à pas

Paul Pitiot, ING 3 Option IA Groupe B

## Objectif retenu

Partir d'une moyenne des tokens précédents et arriver à une tête d'auto-attention dont les poids viennent de `Q`, `K` et `V`. Les quatre versions du sujet : 1 boucles (moyenne causale), 2 produit matriciel (poids fixes), 3 masque et softmax (poids normalisés), 4 self-attention (poids calculés).

`x (B, T, C) → Q, K, V (B, T, dh) → wei = softmax(masque(Q·Kᵀ / √dh)) (B, T, T) → out = wei · V (B, T, dh)`

Invariant à chaque étape : une ligne de `wei` somme à 1 et ne pèse jamais le futur.

Livrable : les quatre versions, les tests `torch.allclose`, la matrice `wei[0]` et une explication des formes `B, T, C, dh`. Aucun entraînement : les projections restent aléatoires, je vérifie le mécanisme. Les graines sont fixées et aucun fichier n'est lu ni écrit.

## 0. Environnement et config

Constantes en majuscules ; graines du sujet (42 pour l'étape 1, 1337 ensuite).

In [1]:
import platform

import torch
import torch.nn as nn
from torch.nn import functional as F

# ---- config ----
SEED_ETAPE_1 = 42                # graine de l'étape 1 dans le sujet
SEED = 1337                      # graine de toutes les autres étapes aléatoires
B, T = 4, 8                      # B séquences indépendantes de T tokens
C_JOUET = 2                      # étapes 2 à 4 : 2 canaux, pour lire les valeurs à la main
C = 32                           # étape 5 : dimension de l'embedding d'un token
HEAD_SIZE = 16                   # dh : dimension de Q, K et V dans une tête
DH_VARIANTE = (4, 16, 64, 256)   # étape 6, variante : valeurs de dh comparées
N_SEQ_VARIANTE = 64              # étape 6, variante : séquences tirées par valeur de dh

print(f"python {platform.python_version()} | torch {torch.__version__}")
print(f"SEED_ETAPE_1 = {SEED_ETAPE_1} | SEED = {SEED}")
print(f"B = {B} | T = {T} | C_JOUET = {C_JOUET} | C = {C} | HEAD_SIZE = {HEAD_SIZE}")

python 3.11.15 | torch 2.14.0
SEED_ETAPE_1 = 42 | SEED = 1337
B = 4 | T = 8 | C_JOUET = 2 | C = 32 | HEAD_SIZE = 16


## 1. Étape 1 : agréger par produit matriciel

`c = a @ b` avec `a` triangulaire inférieure normalisée par ligne : une ligne de `a` choisit et pondère des vecteurs de `b`.

In [2]:
torch.manual_seed(SEED_ETAPE_1)

a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b

print(a)
print(b)
print(c)

# contrôles : formes, somme des lignes, et c[t] = moyenne des t+1 premières lignes de b
assert a.shape == (3, 3) and b.shape == (3, 2) and c.shape == (3, 2)
assert torch.allclose(a.sum(dim=1), torch.ones(3))
for t in range(3):
    assert torch.allclose(c[t], b[:t + 1].mean(dim=0))

# variante : sans la normalisation, a @ b donne des sommes cumulées (l'échelle croît avec t)
a_brute = torch.tril(torch.ones(3, 3))
print(a_brute @ b)
assert torch.allclose(a_brute @ b, b.cumsum(dim=0))

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])
tensor([[ 2.,  7.],
        [ 8., 11.],
        [14., 16.]])


**Questions**

- Sans Python, calculez `c2`. Pourquoi la somme de chaque ligne de `a` vaut-elle 1 ?

Réponses :

- Sans Python, `c2` est la moyenne des 3 lignes de `b` : `((2+6+6)/3, (7+4+5)/3) ≈ (4,6667 ; 5,3333)` en indexation Python (3e ligne). Avec la numérotation du sujet (`t` de 1 à 3), `c_2` est la 2e ligne, `(4 ; 5,5)`.
- La somme de chaque ligne de `a` vaut 1 parce que chaque ligne est divisée par sa somme : `a @ b` calcule alors une moyenne pondérée.

> *Commentaire - 1 :* les `assert` vérifient que `c[t]` est la moyenne des `t+1` premières lignes de `b`. Sans normalisation, `a @ b` donne des sommes cumulées (dernière ligne `(14 ; 16)`, trois fois la moyenne), donc l'échelle dépendrait de `t`.

## 2. Étape 2 : moyenner uniquement le passé

Boucles sur `B` et `T` : pour chaque séquence et chaque instant `t`, moyenne des tokens jusqu'à `t` inclus. C'est la version de référence, lente mais évidente.

In [3]:
torch.manual_seed(SEED)
x = torch.randn(B, T, C_JOUET)

xbow = torch.zeros((B, T, C_JOUET))
for ib in range(B):
    for t in range(T):
        xprev = x[ib, :t + 1]        # (t+1, C_JOUET)
        xbow[ib, t] = torch.mean(xprev, dim=0)

print(xbow[0])

# formes de xprev demandées par le sujet
for t in (0, 3, 7):
    print(f"t = {t} : xprev {tuple(x[0, :t + 1].shape)}")

# contrôles : un token seul est sa propre moyenne, le dernier voit toute la séquence
assert xbow.shape == x.shape
assert torch.allclose(xbow[:, 0], x[:, 0])
assert torch.allclose(xbow[:, -1], x.mean(dim=1))

# variante : moyenner sur dim=1 au lieu de dim=0 moyenne les canaux au lieu des tokens
print(torch.mean(x[0, :4], dim=1).shape, "au lieu de", torch.mean(x[0, :4], dim=0).shape)

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])
t = 0 : xprev (1, 2)
t = 3 : xprev (4, 2)
t = 7 : xprev (8, 2)
torch.Size([4]) au lieu de torch.Size([2])


**Questions**

- Donnez les formes de `xprev` pour `t=0`, `t=3` et `t=7`. Pourquoi la moyenne porte-t-elle sur `dim=0` ?

Réponses :

- `xprev = x[b, :t+1]` a la forme `(t+1, C)` : `(1, 2)` pour `t=0`, `(4, 2)` pour `t=3` et `(8, 2)` pour `t=7`.
- La moyenne porte sur `dim=0` parce que c'est l'axe des tokens passés : elle moyenne les tokens et non les canaux, et donne un vecteur `(C,)`.

> *Commentaire - 2 :* le premier token est sa propre moyenne et le dernier voit toute la séquence (`assert`). Avec `dim=1`, `torch.mean` renverrait `(4,)` au lieu de `(2,)`, c'est-à-dire une moyenne sur les canaux et non sur les tokens.

## 3. Étapes 3 et 4 : vectoriser

La matrice triangulaire normalisée reproduit la moyenne causale ; le masque `-inf` suivi de softmax donne les mêmes poids. Les deux versions sont comparées à la version à boucles par `torch.allclose`.

In [4]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
wei_moyenne = wei                    # gardé pour comparer les poids de l'étape suivante

xbow2 = wei @ x
print(torch.allclose(
    xbow, xbow2, atol=1e-6
))

True


In [5]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

xbow3 = wei @ x
print(torch.allclose(
    xbow, xbow3, atol=1e-6
))

# les poids eux-mêmes sont identiques, pas seulement le résultat
assert torch.allclose(wei, wei_moyenne, atol=1e-6)
assert torch.allclose(wei.sum(dim=1), torch.ones(T))

# variante : avec des scores non nuls, le masque et la normalisation tiennent encore, la moyenne non
torch.manual_seed(SEED)
scores = torch.randn(T, T).masked_fill(tril == 0, float('-inf'))
wei_variante = F.softmax(scores, dim=-1)
assert torch.allclose(wei_variante.sum(dim=1), torch.ones(T))
assert (wei_variante[tril == 0] == 0).all()
print(wei_variante[:3])

True
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3086, 0.6914, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6365, 0.1437, 0.2198, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]])


**Questions**

- Pourquoi `wei @ x` fonctionne-t-il avec `(T, T)` et `(B, T, C)` ? Que devient softmax aux emplacements `-inf` ?

Réponses :

- `wei @ x` fonctionne parce que PyTorch diffuse `wei` `(T, T)` sur le batch : il calcule `wei @ x[b]` pour chaque `b` et renvoie `(B, T, C)`.
- Softmax donne un poids exactement nul aux emplacements `-inf`, car `exp(-inf) = 0`, et chaque ligne somme toujours à 1.

> *Commentaire - 3 :* les deux versions redonnent `xbow` (`True`) et leurs poids sont égaux (`assert`). Avec des scores aléatoires, la ligne 1 vaut `(0,31 ; 0,69)` au lieu de `(0,5 ; 0,5)` : c'est ce qui sépare la moyenne de l'attention.

## 4. Étape 5 : calculer des poids d'attention

Trois projections `C → dh` (`q`, `k`, `v`), `q @ kᵀ`, échelle, masque, softmax, puis `wei @ v`.

In [6]:
torch.manual_seed(SEED)
x = torch.randn(B, T, C)
head_size = HEAD_SIZE
tril = torch.tril(torch.ones(T, T))   # recalculé ici : ne dépend pas de la cellule précédente

key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)                        # (B, T, dh)
q = query(x)                      # (B, T, dh)
wei = q @ k.transpose(-2, -1)     # (B, T, T)
wei = wei * head_size**-0.5       # mise à l'échelle
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)                      # (B, T, dh)
out = wei @ v                     # (B, T, dh)

print(wei.shape, out.shape)
print(wei[0])

assert k.shape == q.shape == v.shape == (B, T, head_size)
assert wei.shape == (B, T, T) and out.shape == (B, T, head_size)

torch.Size([4, 8, 8]) torch.Size([4, 8, 16])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3966, 0.6034, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3069, 0.2892, 0.4039, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3233, 0.2175, 0.2443, 0.2149, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1479, 0.2034, 0.1663, 0.1455, 0.3369, 0.0000, 0.0000, 0.0000],
        [0.1259, 0.2490, 0.1324, 0.1062, 0.3141, 0.0724, 0.0000, 0.0000],
        [0.1598, 0.1990, 0.1140, 0.1125, 0.1418, 0.1669, 0.1061, 0.0000],
        [0.0845, 0.1197, 0.1078, 0.1537, 0.1086, 0.1146, 0.1558, 0.1553]],
       grad_fn=<SelectBackward0>)


**Questions**

- Quelle ligne de `wei[0]` correspond au dernier token ? Pourquoi la sortie mélange-t-elle `v` et non `k` ?

Réponses :

- La ligne `wei[0, 7]`, la dernière, correspond au dernier token : c'est la seule sans position masquée, elle pèse les 8 tokens.
- La sortie mélange `v` parce que `q` et `k` servent seulement à calculer les poids, alors que `v` porte le contenu transmis.

> *Commentaire - 4 :* les formes sont `(4, 8, 16)` pour `q`, `k`, `v` et `out`, et `(4, 8, 8)` pour `wei` (`assert`). La ligne 0 de `wei[0]` vaut `[1, 0, …]` et la ligne 1 vaut `(0,40 ; 0,60)`. Les poids sont proches de l'uniforme parce que les projections ne sont pas entraînées.

## 5. Étape 6 : stabiliser softmax

La variance de `q · k` croît avec `dh` ; diviser par `√dh` garde les scores dans une zone où softmax reste informatif.

In [7]:
torch.manual_seed(SEED)
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)

raw = q @ k.transpose(-2, -1)
scaled = raw * head_size**-0.5

print(raw.var(), scaled.var())

z = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])
print(torch.softmax(z, dim=-1))
print(torch.softmax(z * 8, dim=-1))

tensor(18.4436) tensor(1.1527)
tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])
tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])


> *Commentaire - 5 :* la variance des scores bruts vaut 18,44 pour `dh = 16` (théorie 16) et 1,15 après l'échelle (théorie 1). Sur l'exemple à 5 valeurs, multiplier les scores par 8 fait passer le maximum de softmax de 0,29 à 0,80 : un token domine.

## 6. Bilan du TP 2

Vérification finale sur la tête de la partie 4.

In [8]:
print(wei[0])
print(wei.sum(dim=-1))
assert torch.allclose(wei.sum(dim=-1), torch.ones(B, T), atol=1e-6)

future_mask = ~torch.tril(torch.ones(T, T)).bool()
assert (wei[:, future_mask] == 0).all()

print(out.shape)
assert out.shape == (B, T, head_size)

print("OK : lignes de wei normalisées à 1, aucun poids vers le futur, forme finale (B, T, head_size).")

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3966, 0.6034, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3069, 0.2892, 0.4039, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3233, 0.2175, 0.2443, 0.2149, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1479, 0.2034, 0.1663, 0.1455, 0.3369, 0.0000, 0.0000, 0.0000],
        [0.1259, 0.2490, 0.1324, 0.1062, 0.3141, 0.0724, 0.0000, 0.0000],
        [0.1598, 0.1990, 0.1140, 0.1125, 0.1418, 0.1669, 0.1061, 0.0000],
        [0.0845, 0.1197, 0.1078, 0.1537, 0.1086, 0.1146, 0.1558, 0.1553]],
       grad_fn=<SelectBackward0>)
tensor([[1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000]],
       grad_fn=<SumBackward1>)
torch.Size([4, 8, 16])
OK : lignes de wei nor

**Questions**

- Somme des lignes de `wei`, absence de poids vers le futur, forme finale ? Rôle distinct de `Q`, `K` et `V` ?

Réponses :

- Chaque ligne de `wei` somme à 1, aucun poids ne porte sur le futur (`assert`) et `out` a la forme `(B, T, dh)`.
- `Q` représente ce que le token courant cherche. `K` représente ce que chaque token propose, et `q @ kᵀ` les compare. `V` est le contenu réellement agrégé avec ces poids.

Formes : `x (B, T, C)` → `q, k, v (B, T, dh)` → `wei (B, T, T)` → `out (B, T, dh)`.

| symbole | valeur ici | sens |
|---|---|---|
| `B` | 4 | séquences indépendantes |
| `T` | 8 | positions par séquence |
| `C` | 32 | dimension de l'embedding d'un token |
| `dh` | 16 | dimension de `Q`, `K` et `V` dans une tête |

> *Commentaire - 6 :* les `assert` passent : les lignes somment à 1, les 4 × 28 poids au-dessus de la diagonale sont nuls et `out` a la forme `(4, 8, 16)`. Cette vérification porte sur le mécanisme avec des projections aléatoires, pas sur la qualité d'une tête entraînée.

## 7. Analyse

Quatre vérifications : reproductibilité, échelle (variante sur `dh`), causalité, formes.

In [9]:
def construire_tete(seed, dh, c=C):
    # x puis key, query, value, tirés dans le même ordre que la partie 4
    torch.manual_seed(seed)
    x = torch.randn(B, T, c)
    key, query, value = (nn.Linear(c, dh, bias=False) for _ in range(3))
    return x, key, query, value


def tete(x, key, query, value, causal=True):
    dh = key.out_features
    scores = query(x) @ key(x).transpose(-2, -1) * dh**-0.5
    if causal:
        scores = scores.masked_fill(torch.tril(torch.ones(T, T)) == 0, float('-inf'))
    wei = F.softmax(scores, dim=-1)
    return wei, wei @ value(x)


# 1) reproductibilité
wei_1, out_1 = tete(*construire_tete(SEED, HEAD_SIZE))
wei_2, out_2 = tete(*construire_tete(SEED, HEAD_SIZE))
assert torch.equal(wei_1, wei_2) and torch.equal(out_1, out_2)             # même graine : identique
assert torch.allclose(wei_1, wei, atol=1e-6) and torch.allclose(out_1, out, atol=1e-6)   # = partie 4

wei_3, out_3 = tete(*construire_tete(SEED + 1, HEAD_SIZE))
assert not torch.allclose(wei_1, wei_3)                                    # autre graine : autres poids
assert torch.allclose(wei_3.sum(dim=-1), torch.ones(B, T), atol=1e-6)      # mêmes invariants
assert (wei_3[:, ~torch.tril(torch.ones(T, T)).bool()] == 0).all()
print(f"même graine : écart max {(wei_1 - wei_2).abs().max().item():.1e} | graine + 1 : écart max sur wei {(wei_1 - wei_3).abs().max().item():.3f}")

même graine : écart max 0.0e+00 | graine + 1 : écart max sur wei 0.276


In [10]:
# 2) variante : la variance des scores bruts suit dh, la mise à l'échelle la ramène à 1
torch.manual_seed(SEED)
print("dh    var brute  var mise à l'échelle  poids max moyen (sans / avec échelle)")
for dh in DH_VARIANTE:
    qs = torch.randn(N_SEQ_VARIANTE, T, dh)
    ks = torch.randn(N_SEQ_VARIANTE, T, dh)
    scores = qs @ ks.transpose(-2, -1)
    scores_ech = scores * dh**-0.5
    pmax = F.softmax(scores, dim=-1).max(dim=-1).values.mean()
    pmax_ech = F.softmax(scores_ech, dim=-1).max(dim=-1).values.mean()
    print(f"{dh:<4} {scores.var():>10.2f} {scores_ech.var():>20.2f}   {pmax:.3f} / {pmax_ech:.3f}")

dh    var brute  var mise à l'échelle  poids max moyen (sans / avec échelle)
4          3.94                 0.99   0.535 / 0.344
16        15.71                 0.98   0.755 / 0.358
64        63.38                 0.99   0.874 / 0.357
256      247.07                 0.97   0.936 / 0.357


In [11]:
# 3) test de causalité : on change les tokens 5 à 7, les sorties des positions 0 à 4 ne doivent pas bouger
x, key, query, value = construire_tete(SEED, HEAD_SIZE)
x_modif = x.clone()
torch.manual_seed(SEED + 1)
x_modif[:, 5:] = torch.randn(B, T - 5, C)

for causal in (True, False):
    _, out_a = tete(x, key, query, value, causal)
    _, out_b = tete(x_modif, key, query, value, causal)
    print(f"{'avec' if causal else 'sans'} masque : positions 0-4 inchangées = {torch.allclose(out_a[:, :5], out_b[:, :5], atol=1e-6)}"
          f" | écart max {(out_a[:, :5] - out_b[:, :5]).abs().max().item():.1e}")

avec masque : positions 0-4 inchangées = True | écart max 0.0e+00
sans masque : positions 0-4 inchangées = False | écart max 5.7e-01


In [12]:
# 4) variante de forme : dh ne change que la dernière dimension de out
for dh in (8, 16, 32):
    w, o = tete(*construire_tete(SEED, dh))
    print(f"dh = {dh:<3} wei {tuple(w.shape)} out {tuple(o.shape)}")
    assert w.shape == (B, T, T) and o.shape == (B, T, dh)

dh = 8   wei (4, 8, 8) out (4, 8, 8)
dh = 16  wei (4, 8, 8) out (4, 8, 16)
dh = 32  wei (4, 8, 8) out (4, 8, 32)


> *Commentaire - 7 :*
> - Reproductibilité : avec la même graine, `wei` et `out` sont identiques (écart 0) et identiques à la partie 4. Avec la graine + 1, les poids changent (écart 0,276) mais les invariants restent vrais.
> - Échelle : la variance brute vaut 3,94, 15,71, 63,38 et 247,07 pour `dh = 4, 16, 64, 256`, et `1/√dh` la ramène entre 0,97 et 0,99. Le poids maximal moyen de softmax passe de 0,535 à 0,936 sans échelle, et reste à 0,34-0,36 avec.
> - Causalité : changer les tokens 5 à 7 ne modifie pas les sorties 0 à 4 avec le masque (écart 0), mais les modifie sans masque (écart 0,57).
> - Formes : `dh = 8, 16, 32` ne change que la dernière dimension de `out`.
>
> Limites : le test porte sur un petit batch aléatoire, une seule tête et des projections non entraînées.

## 8. Consignes du sujet

| Consigne | Où |
|---|---|
| Étape 1 : produit matriciel, question sur `c2` et la somme des lignes | partie 1 |
| Étape 2 : boucles, formes de `xprev`, `dim=0` | partie 2 |
| Étapes 3 et 4 : vectorisation, `torch.allclose`, `wei @ x` et `-inf` | partie 3 |
| Étape 5 : `Q`, `K`, `V`, matrice `wei[0]`, dernier token, rôle de `v` | partie 4 |
| Étape 6 : stabiliser softmax (`1/√dh`) | partie 5, variante en partie 7 |
| Bilan : somme des lignes, absence de futur, forme finale, rôle de `Q`, `K`, `V`, formes `B, T, C, dh` | partie 6 |
| Tests `torch.allclose` | parties 3 et 7 |
| Formes vérifiées, rôle expliqué, variante testée à chaque étape | `assert` et variante dans chaque partie |